In [2]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer

PATH_S1 = '/kaggle/input/datasets/bintangryan/orf-cleaned-dataset/train_20.csv'
df = pd.read_csv(PATH_S1)

MODEL_NAME = "indobenchmark/indobert-base-p2" 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

text_cols = ['title_id', 'company_profile_id', 'description_id', 'requirements_id', 'benefits_id']

# Fungsi untuk menghitung panjang token per fitur teks (untuk menentukan max_len)
def get_token_len(text):
    if pd.isna(text): # Handle missing values (NaN)
        return 0
    # Hitung jumlah token setelah di-tokenize (tanpa di-truncate dulu)
    tokens = tokenizer.encode(str(text), add_special_tokens=True, truncation=False)
    return len(tokens)

# Iterasi untuk mencari Mean, P95, dan Max Token
print(f"Menghitung token lengths menggunakan model {MODEL_NAME}...\n")
stats = []

for col in text_cols:
    if col in df.columns:
        print(f"Memproses kolom: {col}...")
        # Apply fungsi ke seluruh baris
        token_lens = df[col].apply(get_token_len)
        
        # Hitung statistik
        mean_len = token_lens.mean()
        p90_len = np.percentile(token_lens, 90)
        max_len = token_lens.max()
        
        stats.append({
            'Column': col,
            'Mean': round(mean_len, 2),
            'P90 (90%)': int(p90_len),
            'Max': int(max_len)
        })
    else:
        print(f"Kolom {col} tidak ditemukan di dataset!")

stats_df = pd.DataFrame(stats)
print("\n=== HASIL STATISTIK PANJANG TOKEN ===")
display(stats_df)

Menghitung token lengths menggunakan model indobenchmark/indobert-base-p2...

Memproses kolom: title_id...
Memproses kolom: company_profile_id...
Memproses kolom: description_id...
Memproses kolom: requirements_id...
Memproses kolom: benefits_id...

=== HASIL STATISTIK PANJANG TOKEN ===


,Column,Mean,P90 (90%),Max
0,title_id,6.36,10,22
1,company_profile_id,101.16,220,661
2,description_id,206.49,402,897
3,requirements_id,99.17,215,810
4,benefits_id,39.17,119,595


In [ ]:
import gc
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold
from scipy.stats import ttest_rel, wilcoxon
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm

# --- ENVIRONMENT SETUP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

# --- DATASET PATHS ---
PATH_S1 = '/kaggle/input/datasets/bintangryan/orf-cleaned-dataset/train_20.csv'
PATH_S2 = '/kaggle/input/datasets/bintangryan/orf-cleaned-dataset/train_40.csv'
PATH_TEST = '/kaggle/input/datasets/bintangryan/orf-cleaned-dataset/test_20f.csv'

# --- HYPERPARAMETERS ---
MAX_LENS = {'title': 16, 'profile': 256, 'desc': 512, 'req': 256, 'ben': 150}
text_cols = ['title_id', 'company_profile_id', 'description_id', 'requirements_id', 'benefits_id']
NUM_FEATURES = [
    'company_profile_id_len', 'description_id_len', 'requirements_id_len', 
    'benefits_id_len', 'text_indo_len', 'has_company_profile_id', 'has_fulltext', 
    'has_company_logo', 'required_education', 'required_experience', 'missing_info_count'
]

print(f"✅ Cell 1: Environment dan konfigurasi hyperparameter berhasil dikunci pada device: {device}")

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, df, tokenizer, max_lengths, numeric_array):
        self.df = df
        self.tokenizer = tokenizer
        self.max_lengths = max_lengths
        self.labels = df['fraudulent'].values
        self.numerics = numeric_array

    def __len__(self):
        return len(self.labels)

    def _tokenize(self, text, max_len):
        return self.tokenizer(str(text), max_length=max_len, padding='max_length', truncation=True, return_tensors='pt')

    def __getitem__(self, item):
        row = self.df.iloc[item]
        t_tok = self._tokenize(row['title_id'], self.max_lengths['title'])
        p_tok = self._tokenize(row['company_profile_id'], self.max_lengths['profile'])
        d_tok = self._tokenize(row['description_id'], self.max_lengths['desc'])
        r_tok = self._tokenize(row['requirements_id'], self.max_lengths['req'])
        b_tok = self._tokenize(row['benefits_id'], self.max_lengths['ben'])

        return {
            't_ids': t_tok['input_ids'].squeeze(0), 't_mask': t_tok['attention_mask'].squeeze(0),
            'p_ids': p_tok['input_ids'].squeeze(0), 'p_mask': p_tok['attention_mask'].squeeze(0),
            'd_ids': d_tok['input_ids'].squeeze(0), 'd_mask': d_tok['attention_mask'].squeeze(0),
            'r_ids': r_tok['input_ids'].squeeze(0), 'r_mask': r_tok['attention_mask'].squeeze(0),
            'b_ids': b_tok['input_ids'].squeeze(0), 'b_mask': b_tok['attention_mask'].squeeze(0),
            'numerics': torch.tensor(self.numerics[item], dtype=torch.float),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

print("✅ Cell 2: Dataset class standar 'CustomDataset' berhasil didefinisikan.")

In [ ]:
class MainModel(nn.Module):
    def __init__(self, model_name, num_extra_features):
        super(MainModel, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.bert.config.hidden_size

        # Multihead Self-Attention antar 5 fitur teks
        self.attention_layer = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        
        # Layer proyeksi linier fitur kontekstual (non-teks)
        self.numeric_layer = nn.Sequential(
            nn.Linear(num_extra_features, 256), 
            nn.LayerNorm(256), 
            nn.ReLU(), 
            nn.Dropout(0.1)
        )

        self.dim_mode_fusion = (self.hidden_size * 5) + 256
        self.dim_mode_text_only = (self.hidden_size * 5)

        # Dua classifier terpisah berdasarkan skenario fitur
        self.classifier_fusion = nn.Sequential(
            nn.Linear(self.dim_mode_fusion, 512), 
            nn.LayerNorm(512), 
            nn.ReLU(), 
            nn.Dropout(0.2), 
            nn.Linear(512, 2)
        )
        self.classifier_text_only = nn.Sequential(
            nn.Linear(self.dim_mode_text_only, 512), 
            nn.LayerNorm(512), 
            nn.ReLU(), 
            nn.Dropout(0.2), 
            nn.Linear(512, 2)
        )

    def forward(self, t_ids, t_mask, p_ids, p_mask, d_ids, d_mask, r_ids, r_mask, b_ids, b_mask, numerics, mode='fusion'):
        # Ekstraksi token pertama [CLS] indeks 0
        o_t = self.bert(input_ids=t_ids, attention_mask=t_mask).last_hidden_state[:, 0, :].unsqueeze(1)
        o_p = self.bert(input_ids=p_ids, attention_mask=p_mask).last_hidden_state[:, 0, :].unsqueeze(1)
        o_d = self.bert(input_ids=d_ids, attention_mask=d_mask).last_hidden_state[:, 0, :].unsqueeze(1)
        o_r = self.bert(input_ids=r_ids, attention_mask=r_mask).last_hidden_state[:, 0, :].unsqueeze(1)
        o_b = self.bert(input_ids=b_ids, attention_mask=b_mask).last_hidden_state[:, 0, :].unsqueeze(1)

        text_embeddings = torch.cat((o_t, o_p, o_d, o_r, o_b), dim=1)
        
        # Pemrosesan Multihead Self-Attention
        attn_output, _ = self.attention_layer(text_embeddings, text_embeddings, text_embeddings)
        text_features = attn_output.reshape(attn_output.shape[0], -1) 

        # Percabangan Skenario Pengujian Berdasarkan Parameter 'mode'
        if mode == 'text_only':
            return self.classifier_text_only(text_features)
        else:
            # Mode Fusion: Fitur teks kontekstual + Fitur numerik terproyeksi
            numeric_features = self.numeric_layer(numerics)
            combined_features = torch.cat((text_features, numeric_features), dim=1)
            return self.classifier_fusion(combined_features)

print("✅ Cell 3: Arsitektur 'MainModel' siap digunakan.")

In [ ]:
def train_fn(model, loader, optimizer, loss_fn, scaler, mode):
    model.train()
    running_loss = 0.0
    loop = tqdm(loader, leave=False)
    for batch_idx, batch in enumerate(loop):
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            logits = model(
                batch['t_ids'].to(device), batch['t_mask'].to(device), 
                batch['p_ids'].to(device), batch['p_mask'].to(device),
                batch['d_ids'].to(device), batch['d_mask'].to(device), 
                batch['r_ids'].to(device), batch['r_mask'].to(device),
                batch['b_ids'].to(device), batch['b_mask'].to(device), 
                batch['numerics'].to(device), mode=mode
            )
            loss = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    return running_loss / len(loader)

def eval_fn(model, loader, tag, mode, device):
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Eval {tag}"):
            with autocast('cuda'):
                logits = model(
                    batch['t_ids'].to(device), batch['t_mask'].to(device), 
                    batch['p_ids'].to(device), batch['p_mask'].to(device),
                    batch['d_ids'].to(device), batch['d_mask'].to(device), 
                    batch['r_ids'].to(device), batch['r_mask'].to(device),
                    batch['b_ids'].to(device), batch['b_mask'].to(device), 
                    batch['numerics'].to(device), mode=mode
                )
            probs = torch.softmax(logits, dim=1)[:, 1]
            y_true.extend(batch['labels'].tolist())
            y_pred.extend(torch.argmax(logits, dim=1).cpu().tolist())
            y_probs.extend(probs.cpu().tolist())

    # Hitung Confusion Matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    # Hitung Metrik
    accuracy    = (tp + tn) / (tp + tn + fp + fn)
    precision   = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall      = tp / (tp + fn) if (tp + fn) != 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    f1          = 2 * precision * recall / (precision + recall) if (precision + recall) != 0 else 0

    # Print Report
    print(f"\n{'═'*70}\n FINAL REPORT: {tag.upper()} \n{'═'*70}")
    print(f"{'Accuracy':<30} | {accuracy:.4f}\n{'F1-Score':<30} | {f1:.4f}\n{'Recall (Sensitivity)':<30} | {recall:.4f}\n{'Specificity (TNR)':<30} | {specificity:.4f}\n{'Precision (PPV)':<30} | {precision:.4f}")
    print(f"{'-'*70}\nMatrix Result: TP={tp}, TN={tn}, FP={fp}, FN={fn}\n{'═'*70}\n")

    # Plot & Save Confusion Matrix
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Negative (0)', 'Positive (1)'], 
                yticklabels=['Negative (0)', 'Positive (1)'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix: {tag}')
    
    # Simpan sebagai file PNG
    plt.savefig(f'confusion_matrix_{tag}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Confusion matrix berhasil disimpan sebagai: confusion_matrix_{tag}.png")

    return pd.DataFrame({'True': y_true, 'Pred': y_pred, 'Prob': y_probs})

In [ ]:
try:
    df_s1 = pd.read_csv(PATH_S1)
    df_s2 = pd.read_csv(PATH_S2)
    df_test = pd.read_csv(PATH_TEST)
    print("✅ Cell 5: Semua file dataset sukses dimuat ke dataframe pandas.")
except Exception as e:
    print(f"❌ Cell 5: Gagal memuat dataset! Error: {e}")

# Imputasi basic handling missing values (NaN)
for df in [df_s1, df_s2, df_test]:
    df[NUM_FEATURES] = df[NUM_FEATURES].fillna(0)
    df[text_cols] = df[text_cols].fillna('')

print("\n" + "="*50)
print(f"{'DATA INTEGRITY PROFILE SUMMARY':^50}")
print("="*50)
for df, label in zip([df_s1, df_s2, df_test], ["Skenario 1 (S1)", "Skenario 2 (S2)", "Test Set Final"]):
    total = len(df)
    fraud = df['fraudulent'].sum()
    print(f"{label:<18} -> Total Data: {total:>5} | Total Fraud: {fraud:>4} ({ (fraud/total)*100 :>4.1f}%)")
print("="*50)

In [ ]:
configs = [
    {"name": "IndoBERT_Benchmark", "m_path": "indobenchmark/indobert-base-p2"},
    {"name": "IndoBERT_IndoLEM", "m_path": "indolem/indobert-base-uncased"},
]
datasets = [{"name": "S1_20", "data": df_s1}, {"name": "S2_40", "data": df_s2}]
experiment_modes = ['text_only', 'fusion'] # Penamaan konsisten 'fusion'

for m_cfg in configs:
    tokenizer = AutoTokenizer.from_pretrained(m_cfg['m_path'])
    for d_cfg in datasets:
        for mode in experiment_modes:
            tag = f"{m_cfg['name']}_{d_cfg['name']}_{mode.upper()}"
            print(f"\nEKSEKUSI TRAINING: {tag.upper()}")
            
            train_loader = DataLoader(CustomDataset(d_cfg['data'], tokenizer, MAX_LENS, d_cfg['data'][NUM_FEATURES].values), batch_size=8, shuffle=True)
            test_loader = DataLoader(CustomDataset(df_test, tokenizer, MAX_LENS, df_test[NUM_FEATURES].values), batch_size=8, shuffle=False)
            
            model = MainModel(m_cfg['m_path'], len(NUM_FEATURES)).to(device)
            optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
            scaler = GradScaler('cuda')
            
            neg_counts = np.sum(d_cfg['data']['fraudulent'].values == 0)
            pos_counts = np.sum(d_cfg['data']['fraudulent'].values == 1)
            loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, float(neg_counts / pos_counts)], dtype=torch.float).to(device))
            
            for epoch in range(4):
                avg_loss = train_fn(model, train_loader, optimizer, loss_fn, scaler, mode)
                print(f"Epoch {epoch+1}/4 Berhasil | Avg Training Loss: {avg_loss:.4f}")
            
            torch.save(model.state_dict(), f"model_{tag}.pth")
            res_df = eval_fn(model, test_loader, tag, mode, device)
            res_df.to_csv(f"results_{tag}.csv", index=False)
            
            del model, train_loader, test_loader
            torch.cuda.empty_cache()
            gc.collect()

print("\n\nSELESAI! SELURUH SKENARIO UTAMA BERHASIL DIEKSEKUSI SECARA BERURUTAN.")

In [ ]:
configs = [
    {"name": "mBERT", "m_path": "bert-base-multilingual-cased"}
]
datasets = [{"name": "S1_20", "data": df_s1}, {"name": "S2_40", "data": df_s2}]
experiment_modes = ['text_only', 'fusion'] # Penamaan konsisten 'fusion'

for m_cfg in configs:
    tokenizer = AutoTokenizer.from_pretrained(m_cfg['m_path'])
    for d_cfg in datasets:
        for mode in experiment_modes:
            tag = f"{m_cfg['name']}_{d_cfg['name']}_{mode.upper()}"
            print(f"\nEKSEKUSI TRAINING: {tag.upper()}")
            
            train_loader = DataLoader(CustomDataset(d_cfg['data'], tokenizer, MAX_LENS, d_cfg['data'][NUM_FEATURES].values), batch_size=8, shuffle=True)
            test_loader = DataLoader(CustomDataset(df_test, tokenizer, MAX_LENS, df_test[NUM_FEATURES].values), batch_size=8, shuffle=False)
            
            model = MainModel(m_cfg['m_path'], len(NUM_FEATURES)).to(device)
            optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
            scaler = GradScaler('cuda')
            
            neg_counts = np.sum(d_cfg['data']['fraudulent'].values == 0)
            pos_counts = np.sum(d_cfg['data']['fraudulent'].values == 1)
            loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, float(neg_counts / pos_counts)], dtype=torch.float).to(device))
            
            for epoch in range(4):
                avg_loss = train_fn(model, train_loader, optimizer, loss_fn, scaler, mode)
                print(f"Epoch {epoch+1}/4 Berhasil | Avg Training Loss: {avg_loss:.4f}")
            
            torch.save(model.state_dict(), f"model_{tag}.pth")
            res_df = eval_fn(model, test_loader, tag, mode)
            res_df.to_csv(f"results_{tag}.csv", index=False)
            
            del model, train_loader, test_loader
            torch.cuda.empty_cache()
            gc.collect()

print("\n\nSELESAI! SELURUH 8 SKENARIO UTAMA BERHASIL DIEKSEKUSI SECARA BERURUTAN.")

# K FOLD CROSS VALIDATION

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 5
df_kfold = df_s1.copy().reset_index(drop=True)
df_kfold['fold'] = -1

# Mengunci random_state agar pembagian fold konsisten dan reproducible (Seed: 42)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

for fold_idx, (_, val_idx) in enumerate(skf.split(df_kfold, df_kfold['fraudulent'])):
    df_kfold.loc[val_idx, 'fold'] = fold_idx

print(f"{'='*60}\nDISTRIBUSI PROPORSI DATA PER BLOK FOLD (DATASET S1)\n{'='*60}")
distribution_report = []
for fold_idx in range(N_SPLITS):
    pure_fold_df = df_kfold[df_kfold.fold == fold_idx]
    fraud_count = pure_fold_df['fraudulent'].sum()
    normal_count = len(pure_fold_df) - fraud_count
    
    distribution_report.append({
        "Blok / Fold": f"Blok Fold {fold_idx + 1}",
        "Ukuran Murni Data Uji": len(pure_fold_df),
        "Jumlah Normal": normal_count,
        "Jumlah Fraud": fraud_count,
        "Rasio Fraud": f"{round(pure_fold_df['fraudulent'].mean() * 100, 2)}%"
    })

df_report = pd.DataFrame(distribution_report)
total_row = pd.DataFrame([{
    "Blok / Fold": "TOTAL DATASET S1",
    "Ukuran Murni Data Uji": df_report["Ukuran Murni Data Uji"].sum(),
    "Jumlah Normal": df_report["Jumlah Normal"].sum(),
    "Jumlah Fraud": df_report["Jumlah Fraud"].sum(),
    "Rasio Fraud": f"{round((df_report['Jumlah Fraud'].sum() / df_report['Ukuran Murni Data Uji'].sum()) * 100, 2)}%"
}])
df_report = pd.concat([df_report, total_row], ignore_index=True)
display(df_report)

print("\n✅ Cell 1: Folding data selesai. Tiap blok murni terverifikasi memiliki distribusi yang sama.")

In [ ]:
import gc
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import confusion_matrix

configs = [
    {"name": "IndoBERT_Benchmark", "m_path": "indobenchmark/indobert-base-p2"},
    {"name": "IndoBERT_IndoLEM", "m_path": "indolem/indobert-base-uncased"}
]
kfold_modes = ['text_only', 'fusion']
all_results = {}

for m_cfg in configs:
    tokenizer = AutoTokenizer.from_pretrained(m_cfg["m_path"])
    for mode in kfold_modes:
        experiment_name = f"{m_cfg['name']}_{mode}"
        print(f"\n{'='*80}\nEXPERIMENT K-FOLD: {experiment_name}\n{'='*80}")
        fold_metrics = []

        for fold_idx in range(N_SPLITS):
            tag = f"{m_cfg['name']}_FOLD_{fold_idx+1}_{mode.upper()}"
            
            # Cetak peta visual balok highlight data fold aktif
            visual_blocks = ["🟥[TEST]" if i == fold_idx else "■[TRAIN]" for i in range(N_SPLITS)]
            print(f"DATA FOLD {fold_idx+1}:  {' | '.join(visual_blocks)}")
            
            # Slicing data murni untuk train dan val berbasis fold_idx
            train_df = df_kfold[df_kfold.fold != fold_idx].reset_index(drop=True)
            val_df = df_kfold[df_kfold.fold == fold_idx].reset_index(drop=True)
            
            train_loader = DataLoader(CustomDataset(train_df, tokenizer, MAX_LENS, train_df[NUM_FEATURES].values), batch_size=8, shuffle=True)
            val_loader = DataLoader(CustomDataset(val_df, tokenizer, MAX_LENS, val_df[NUM_FEATURES].values), batch_size=8, shuffle=False)
            
            model = MainModel(m_cfg["m_path"], len(NUM_FEATURES)).to(device)
            
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
            scaler = GradScaler('cuda')
            
            neg_c = np.sum(train_df['fraudulent'].values == 0)
            pos_c = np.sum(train_df['fraudulent'].values == 1)
            loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, float(neg_c / pos_c)], dtype=torch.float).to(device))

            # Proses pelatihan sepanjang 4 Epoch
            for epoch in range(4):
                avg_loss = train_fn(model, train_loader, optimizer, loss_fn, scaler, mode)
                print(f"   [Fold {fold_idx+1}/5] Epoch {epoch+1}/4 Berhasil | Avg Training Loss: {avg_loss:.4f}")

            # Evaluasi final menggunakan fungsi eval_fn
            res_df = eval_fn(model, val_loader, tag, mode)
            
            # Ekstraksi otomatis metrics untuk komparasi tabel statistik
            y_true, y_pred = res_df['True'].values, res_df['Pred'].values
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
            
            acc = (tp + tn) / (tp + tn + fp + fn)
            prec = tp / (tp + fp) if (tp + fp) != 0 else 0
            rec = tp / (tp + fn) if (tp + fn) != 0 else 0
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) != 0 else 0
            spec = tn / (tn + fp) if (tn + fp) != 0 else 0
            
            fold_metrics.append({
                "Fold": fold_idx + 1, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "Specificity": spec
            })
            
            # Pembersihan total VRAM memori GPU per fold
            del model, train_loader, val_loader, res_df
            torch.cuda.empty_cache()
            gc.collect()

        # Rekap rangkuman statistik akhir skenario ke DataFrame
        report_df = pd.DataFrame(fold_metrics)
        numeric_cols = report_df.drop(columns=["Fold"])
        stats_rows = pd.DataFrame([
            {"Fold": "Mean", **numeric_cols.mean().to_dict()},
            {"Fold": "Std", **numeric_cols.std().to_dict()}
        ])
        report_df = pd.concat([report_df, stats_rows], ignore_index=True)
        
        all_results[experiment_name] = report_df
        report_df.to_csv(f"{experiment_name}_kfold.csv", index=False)
        display(report_df)

print("\n✅ Cell 2: Seluruh eksperimen K-Fold selesai dieksekusi lurus dengan aman.")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, wilcoxon

ALPHA = 0.05
METRICS_TO_TEST = ["Accuracy", "Precision", "Recall", "F1", "Specificity"]

for model_name in ["IndoBERT_Benchmark", "IndoBERT_IndoLEM"]:
    print(f"\n" + "="*90)
    print(f" ANALISIS UJI SIGNIFIKANSI STATISTIK UNTUK MODEL: {model_name.upper()} ")
    print("="*90)
    
    # Menarik baris data fold murni (1-5) dari memori hasil eksekusi Cell 2
    df_text = all_results[f"{model_name}_text_only"].iloc[:N_SPLITS]
    df_fusion = all_results[f"{model_name}_fusion"].iloc[:N_SPLITS]
    
    statistical_report = []
    for metric in METRICS_TO_TEST:
        text_vals = df_text[metric].astype(float).values
        fusion_vals = df_fusion[metric].astype(float).values
        
        # 1. Eksekusi Uji Parametrik (Paired t-test)
        _, p_ttest = ttest_rel(text_vals, fusion_vals)
        sig_ttest = "✅ YA (Signifikan)" if p_ttest < ALPHA else "❌ TIDAK"
        
        # 2. Eksekusi Uji Non-Parametrik (Wilcoxon Signed-Rank Test)
        try:
            _, p_wilcoxon = wilcoxon(text_vals, fusion_vals, zero_method='pratt')
        except ValueError:
            p_wilcoxon = 1.0 # Penanganan jika selisih data bernilai 0 mutlak
        sig_wilcoxon = "✅ YA (Signifikan)" if p_wilcoxon < ALPHA else "❌ TIDAK"
        
        statistical_report.append({
            "Metrik Evaluasi": metric,
            "Mean (Text Only)": f"{text_vals.mean():.4f}",
            "Mean (Fusion)": f"{fusion_vals.mean():.4f}",
            "Selisih (Δ)": f"{(fusion_vals.mean() - text_vals.mean()):+.4f}",
            "p-value (T-Test)": f"{p_ttest:.6f}",
            "Signifikan (T-Test)": sig_ttest,
            "p-value (Wilcoxon)": f"{p_wilcoxon:.6f}",
            "Signifikan (Wilcoxon)": sig_wilcoxon
        })
        
    df_stat_final = pd.DataFrame(statistical_report)
    display(df_stat_final)
    print("-" * 90)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel

ALPHA = 0.05
METRICS_TO_TEST = ["Accuracy", "Precision", "Recall", "F1", "Specificity"]

# ==============================================================================
# DATABASE DATA METRIKS AKTUAL
# ==============================================================================
hardcoded_results = {
    "IndoBERT_Benchmark": {
        "text_only": {
            "Accuracy":    [0.945, 0.952, 0.956, 0.908, 0.947],
            "Precision":   [0.853, 0.864, 0.940, 0.729, 0.822],
            "Recall":      [0.876, 0.903, 0.832, 0.858, 0.938],
            "F1":          [0.865, 0.883, 0.883, 0.789, 0.876],
            "Specificity": [0.962, 0.965, 0.987, 0.920, 0.949]
        },
        "fusion": {
            "Accuracy":    [0.947, 0.952, 0.952, 0.924, 0.959],
            "Precision":   [0.874, 0.898, 0.906, 0.824, 0.875],
            "Recall":      [0.858, 0.858, 0.850, 0.788, 0.929],
            "F1":          [0.866, 0.878, 0.877, 0.805, 0.901],
            "Specificity": [0.969, 0.976, 0.978, 0.958, 0.967]
        }
    },
    "IndoBERT_IndoLEM": {
        "text_only": {
            "Accuracy":    [0.922, 0.940, 0.922, 0.924, 0.945],
            "Precision":   [0.795, 0.976, 0.805, 0.818, 0.848],
            "Recall":      [0.823, 0.717, 0.805, 0.797, 0.885],
            "F1":          [0.809, 0.827, 0.805, 0.807, 0.866],
            "Specificity": [0.947, 0.996, 0.951, 0.956, 0.960]
        },
        "fusion": {
            "Accuracy":    [0.936, 0.945, 0.945, 0.906, 0.943],
            "Precision":   [0.803, 0.848, 0.866, 0.742, 0.926],
            "Recall":      [0.903, 0.885, 0.858, 0.814, 0.779],
            "F1":          [0.850, 0.866, 0.862, 0.776, 0.846],
            "Specificity": [0.945, 0.960, 0.967, 0.929, 0.985]
        }
    }
}

# ==============================================================================
# PROSES ANALISIS STATISTIK (PAIRED T-TEST)
# ==============================================================================
for model_name in ["IndoBERT_Benchmark", "IndoBERT_IndoLEM"]:
    print(f"\n" + "="*80)
    print(f" ANALISIS UJI SIGNIFIKANSI (PAIRED T-TEST): {model_name.upper()} ")
    print("="*80)
    
    statistical_report = []
    for metric in METRICS_TO_TEST:
        text_vals = np.array(hardcoded_results[model_name]["text_only"][metric])
        fusion_vals = np.array(hardcoded_results[model_name]["fusion"][metric])
        
        # Eksekusi Uji Statistik (Paired t-test)
        _, p_ttest = ttest_rel(text_vals, fusion_vals)
        sig_ttest = "Signifikan" if p_ttest < ALPHA else "Tidak Signifikan"
        
        statistical_report.append({
            "Metrik": metric,
            "Mean (Text)": f"{text_vals.mean():.3f}",
            "Mean (Fusion)": f"{fusion_vals.mean():.3f}",
            "Selisih (Δ)": f"{(fusion_vals.mean() - text_vals.mean()):+.3f}",
            "p-value": f"{p_ttest:.3f}",
            "Hasil": sig_ttest
        })
        
    df_stat_final = pd.DataFrame(statistical_report)
    display(df_stat_final)
    print("-" * 80)